<a href="https://colab.research.google.com/github/mohammad-ishtiaque/bangla-bias-research/blob/main/notebooks/day04_baseline_replication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 4 — Exact replication of the Pokkhopat baseline

**Purpose:** reproduce the published numbers of Arnob et al. (GeBNLP 2025, *Pokkhopat*)
using their own code path, before making any claim of our own.

Their pipeline, read from https://github.com/HerWILL-Inc/ACL-2025 :

| step | what they do | file |
|---|---|---|
| word lists | `readlines()`, strip only `\n` | `dataprocess.py` |
| normalise | `normalize(word)` from csebuetnlp/normalizer | `calculate_WEAT_score_and_p_value.py` |
| tokenise | `tokenizer.encode()` — keeps `[CLS]` / `[SEP]` | same |
| pool | `hidden_states[-1].mean(dim=1)` — mean over **all** tokens | same |
| model | `AutoModelForPreTraining`, `output_hidden_states=True` | same |
| WEAT d | cosine, `std(..., unbiased=False)` i.e. ddof=0 | `bias_metrics.py` |
| default test | targets = STEM vs SHAPE, attributes = male vs female terms | `main()` |

Nothing here is our own design. Deviations are introduced only in the final
ablation cell, one at a time, so we can attribute any difference.

## 1. Environment and data

In [ ]:
!pip install -q transformers torch huggingface_hub pandas numpy openpyxl
!pip install -q git+https://github.com/csebuetnlp/normalizer

import os, json, glob
os.makedirs("data", exist_ok=True)

from huggingface_hub import snapshot_download
snapshot_download(repo_id="csebuetnlp/BanglaContextualBias", repo_type="dataset",
                  local_dir="data/BCB", ignore_patterns=["*.zip"])

!git clone -q https://github.com/HerWILL-Inc/ACL-2025.git data/Pokkhopat
!git clone -q https://github.com/csebuetnlp/BanglaContextualBias.git data/BCB-code
print("data ready")

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 5.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 6.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 31 files:   0%|          | 0/31 [00:00<?, ?it/s]

data ready


## 2. Their published numbers — the target we must hit

In [ ]:
import pandas as pd

for f in ["data/Pokkhopat/plots/plotting.xlsx",
          "data/Pokkhopat/dataset/OOV analysis.xlsx",
          "data/Pokkhopat/models/Models.xlsx"]:
    print("\n" + "="*70); print(f); print("="*70)
    try:
        xl = pd.ExcelFile(f)
        for s in xl.sheet_names:
            print(f"\n--- sheet: {s} ---")
            print(xl.parse(s).to_string())
    except Exception as e:
        print("ERROR:", type(e).__name__, e)


data/Pokkhopat/plots/plotting.xlsx

--- sheet: Sheet1 ---
        Male    Female                                     Model  Unnamed: 3  Unnamed: 4  Unnamed: 5  Unnamed: 6  Unnamed: 7  Unnamed: 8  Unnamed: 9  Unnamed: 10  Unnamed: 11  Unnamed: 12  Unnamed: 13  Unnamed: 14 Unnamed: 15 Unnamed: 16  Unnamed: 17  Unnamed: 18  Unnamed: 19  Unnamed: 20  Unnamed: 21 Unnamed: 22
0     0.8586    0.8607                     csebuetnlp/banglabert         NaN         NaN         NaN         NaN         NaN         NaN         NaN          NaN          NaN          NaN          NaN          NaN         NaN         NaN          NaN          NaN          NaN          NaN          NaN         NaN
1      0.586    0.5875                    saiful9379/Bangla_GPT2         NaN         NaN         NaN         NaN         NaN         NaN         NaN          NaN          NaN          NaN          NaN          NaN         NaN         NaN          NaN          NaN          NaN          NaN          NaN         

## 3. Their code, transcribed

`pok_embedding` mirrors `get_embeddings()`; `pok_weat` mirrors
`WordEmbeddingAssociationTest.__call__`; `pok_wordlist` mirrors `ProcessData`.

In [ ]:
import torch, numpy as np
from transformers import AutoModelForPreTraining, AutoTokenizer
from normalizer import normalize

torch.set_grad_enabled(False)
torch.set_default_dtype(torch.float64)      # they set this globally


def pok_wordlist(path):
    """dataprocess.py: readlines(), strip newline only. Multi-word terms stay whole."""
    return [w.replace("\n", "") for w in open(path, encoding="UTF-8").readlines()]


def pok_embedding(model, tokenizer, word, use_normalizer=True, drop_special=False):
    """get_embeddings() from calculate_WEAT_score_and_p_value.py.

    use_normalizer / drop_special exist only for the ablation in section 6.
    Baseline = (True, False).
    """
    if use_normalizer:
        word = normalize(word)
    ids = tokenizer.encode(word, return_tensors="pt")
    out = model(ids)
    h = out["hidden_states"][-1]            # last encoder layer
    if drop_special:
        h = h[:, 1:-1, :]
    return h.mean(dim=1)                    # mean pooling over tokens


def pok_weat(X, Y, A, B, test_stat=False, ddof0=True):
    """WordEmbeddingAssociationTest from bias_metrics.py."""
    f = lambda t: torch.nn.functional.normalize(t.flatten(end_dim=-2), p=2, dim=-1)
    X, Y, A, B = f(X), f(Y), f(A), f(B)
    XA, XB = torch.mm(X, A.t()), torch.mm(X, B.t())
    YA, YB = torch.mm(Y, A.t()), torch.mm(Y, B.t())
    sX = XA.mean(-1) - XB.mean(-1)
    sY = YA.mean(-1) - YB.mean(-1)
    s  = sX.mean() - sY.mean()
    if test_stat:
        return s
    S = torch.cat([XA, YA]).mean(-1) - torch.cat([XB, YB]).mean(-1)
    return s / torch.std(S, unbiased=ddof0 is False)


def embed_all(model, tokenizer, words, **kw):
    return torch.stack([pok_embedding(model, tokenizer, w, **kw) for w in words])

## 4. Word lists, loaded their way

In [ ]:
D = "data/Pokkhopat/dataset"
male   = pok_wordlist(f"{D}/male.txt")
female = pok_wordlist(f"{D}/female.txt")
stem   = pok_wordlist(f"{D}/stem.txt")
shape  = pok_wordlist(f"{D}/shape.txt")

for n, v in [("male", male), ("female", female), ("stem", stem), ("shape", shape)]:
    print(f"{n:7} n={len(v):4}  multi-word={sum(1 for w in v if ' ' in w):3}  blank={sum(1 for w in v if not w.strip())}")

print("\nNOTE: their main() uses targets = STEM vs SHAPE, attributes = male vs female.")
print("Unequal target sizes also trigger the permutation-split issue in their p_test")
print("(X_hat = perm[0:len(X)], Y_hat = perm[len(Y):] drops the middle items).")
print("That affects their p-values only; the effect size d uses the full sets.")

male    n=  57  multi-word= 17  blank=0
female  n=  56  multi-word= 16  blank=0
stem    n=  48  multi-word= 24  blank=0
shape   n=  77  multi-word= 35  blank=0

NOTE: their main() uses targets = STEM vs SHAPE, attributes = male vs female.
Unequal target sizes also trigger the permutation-split issue in their p_test
(X_hat = perm[0:len(X)], Y_hat = perm[len(Y):] drops the middle items).
That affects their p-values only; the effect size d uses the full sets.


## 5. Replication run

In [ ]:
import gc

# Encoder models from their modelnames list (GPT2 / T5 entries need from_tf and are
# handled separately; start with the BERT-family subset for speed).
MODELS = [
    "csebuetnlp/banglabert",
    "csebuetnlp/banglishbert",
    "sagorsarker/bangla-bert-base",
    "Kowsher/bangla-bert",
    "neuropark/sahajBERT",
]

rows = []
for name in MODELS:
    print(f"\n{name} ...", end=" ")
    try:
        model = AutoModelForPreTraining.from_pretrained(
            name, output_hidden_states=True, output_attentions=True,
            cache_dir="cache/models/transformers")
        tok = AutoTokenizer.from_pretrained(name, cache_dir="cache/models/transformers")
    except Exception as e:
        print("FAILED", str(e)[:100]); continue

    X = embed_all(model, tok, stem)
    Y = embed_all(model, tok, shape)
    A = embed_all(model, tok, male)
    B = embed_all(model, tok, female)

    d = pok_weat(X, Y, A, B).item()
    rows.append({"model": name, "d_replicated": round(d, 4)})
    print(f"d = {d:+.4f}")

    del model, tok, X, Y, A, B
    gc.collect(); torch.cuda.empty_cache()

rep = pd.DataFrame(rows)
print("\n", rep.to_string(index=False))
rep.to_csv("replication_weat.csv", index=False)


csebuetnlp/banglabert ... 

config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  443MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/1.72M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/322 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 72.4MB            

model.safetensors: downloading bytes:           |  0.00B            

d = +0.1323

                        model  d_replicated
       csebuetnlp/banglabert       -0.1547
     csebuetnlp/banglishbert       -0.0917
sagorsarker/bangla-bert-base       -0.8031
         Kowsher/bangla-bert       -0.2071
         neuropark/sahajBERT        0.1323


## 6. Comparison

Fill `published` from section 2 (or from Table in the paper), then judge.
A replication is usually accepted if the sign matches and |difference| is small
relative to the spread across models.

In [ ]:
# published = {"csebuetnlp/banglabert": ..., ...}   # <-- fill from section 2
published = {}

if published:
    rep["d_published"] = rep["model"].map(published)
    rep["diff"] = (rep["d_replicated"] - rep["d_published"]).round(4)
    rep["sign_match"] = np.sign(rep["d_replicated"]) == np.sign(rep["d_published"])
    print(rep.to_string(index=False))
    rep.to_csv("replication_comparison.csv", index=False)
else:
    print("published numbers not filled in yet -- do section 2 first")

published numbers not filled in yet -- do section 2 first


## 7. Ablation — which of our earlier deviations mattered

Only run this **after** section 6 shows the replication holds. Each row changes
exactly one thing away from the baseline, so any movement in `d` is attributable.

In [ ]:
VARIANTS = {
    "baseline (their code)":      dict(use_normalizer=True,  drop_special=False, ddof0=True),
    "no normalizer":              dict(use_normalizer=False, drop_special=False, ddof0=True),
    "drop [CLS]/[SEP]":           dict(use_normalizer=True,  drop_special=True,  ddof0=True),
    "no norm + drop special":     dict(use_normalizer=False, drop_special=True,  ddof0=True),
    "ddof=1":                     dict(use_normalizer=True,  drop_special=False, ddof0=False),
}

abl = []
for name in MODELS:
    model = AutoModelForPreTraining.from_pretrained(
        name, output_hidden_states=True, cache_dir="cache/models/transformers")
    tok = AutoTokenizer.from_pretrained(name, cache_dir="cache/models/transformers")
    for label, cfg in VARIANTS.items():
        ekw = {k: v for k, v in cfg.items() if k != "ddof0"}
        X = embed_all(model, tok, stem,   **ekw)
        Y = embed_all(model, tok, shape,  **ekw)
        A = embed_all(model, tok, male,   **ekw)
        B = embed_all(model, tok, female, **ekw)
        d = pok_weat(X, Y, A, B, ddof0=cfg["ddof0"]).item()
        abl.append({"model": name.split("/")[-1], "variant": label, "d": round(d, 4)})
        print(f"{name.split('/')[-1]:20} {label:24} d = {d:+.4f}")
    del model, tok; gc.collect(); torch.cuda.empty_cache()

abl_df = pd.DataFrame(abl)
print("\n", abl_df.pivot(index="model", columns="variant", values="d").to_string())
abl_df.to_csv("ablation.csv", index=False)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

banglabert           baseline (their code)    d = -0.1547
banglabert           no normalizer            d = -0.2794
banglabert           drop [CLS]/[SEP]         d = +0.0685
banglabert           no norm + drop special   d = -0.0009
banglabert           ddof=1                   d = -0.1540


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

banglishbert         baseline (their code)    d = -0.0917
banglishbert         no normalizer            d = -0.1054
banglishbert         drop [CLS]/[SEP]         d = +0.1117
banglishbert         no norm + drop special   d = +0.1049
banglishbert         ddof=1                   d = -0.0913


Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

[transformers] BertForPreTraining LOAD REPORT from: Kowsher/bangla-bert
Key                         | Status  | 
----------------------------+---------+-
cls.seq_relationship.weight | MISSING | 
cls.seq_relationship.bias   | MISSING | 
bert.pooler.dense.weight    | MISSING | 
bert.pooler.dense.bias      | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


bangla-bert          baseline (their code)    d = -0.2071
bangla-bert          no normalizer            d = -0.2071
bangla-bert          drop [CLS]/[SEP]         d = -0.2679
bangla-bert          no norm + drop special   d = -0.2679
bangla-bert          ddof=1                   d = -0.2063


Loading weights:   0%|          | 0/34 [00:00<?, ?it/s]

sahajBERT            baseline (their code)    d = +0.1323
sahajBERT            no normalizer            d = +0.1323
sahajBERT            drop [CLS]/[SEP]         d = +0.2191
sahajBERT            no norm + drop special   d = +0.2191
sahajBERT            ddof=1                   d = +0.1318

 variant           baseline (their code)  ddof=1  drop [CLS]/[SEP]  no norm + drop special  no normalizer
model                                                                                                   
bangla-bert                     -0.2071 -0.2063           -0.2679                 -0.2679        -0.2071
bangla-bert-base                -0.8031 -0.7999           -0.7902                 -0.7902        -0.8031
banglabert                      -0.1547 -0.1540            0.0685                 -0.0009        -0.2794
banglishbert                    -0.0917 -0.0913            0.1117                  0.1049        -0.1054
sahajBERT                        0.1323  0.1318            0.2191             

## 8. Save

Download immediately — the Colab VM is discarded when the runtime resets.

In [ ]:
from google.colab import files
import os
for f in ["replication_weat.csv", "replication_comparison.csv", "ablation.csv"]:
    if os.path.exists(f):
        print(f, pd.read_csv(f).shape)
        files.download(f)

replication_weat.csv (5, 2)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

ablation.csv (25, 3)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
def pok_cosmetric(T1, T2, AB):
    """CosMetric from bias_metrics.py — ওদের plotting.xlsx এটাই"""
    m1 = torch.nn.functional.normalize(T1.flatten(end_dim=-2).mean(dim=0), p=2, dim=-1)
    m2 = torch.nn.functional.normalize(T2.flatten(end_dim=-2).mean(dim=0), p=2, dim=-1)
    AB = torch.nn.functional.normalize(AB.flatten(end_dim=-2), p=2, dim=-1)
    cos = torch.nn.CosineSimilarity(dim=1, eps=1e-6)
    return cos(AB, m1).mean(0).item(), cos(AB, m2).mean(0).item()

# ১. তালিকার পুনরাবৃত্তি আগে দেখো
for n, v in [("male",male),("female",female),("stem",stem),("shape",shape)]:
    print(f"{n:7} lines={len(v):3}  unique={len(set(v)):3}  dup={len(v)-len(set(v)):3}")

# ২. ওদের প্রকাশিত cosine
PUB = {  # (STEM_male, STEM_female, SHAPE_male, SHAPE_female)
 "csebuetnlp/banglabert":        (0.8586, 0.8607, 0.888396, 0.889532),
 "neuropark/sahajBERT":          (0.7008, 0.6941, 0.729031, 0.724120),
 "Kowsher/bangla-bert":          (0.5034, 0.4962, 0.522467, 0.512392),
 "csebuetnlp/banglishbert":      (0.9161, 0.9189, 0.936249, 0.938690),
 "sagorsarker/bangla-bert-base": (0.5416, 0.5896, 0.559090, 0.586253),
}

rows=[]
for name in MODELS:
    model = AutoModelForPreTraining.from_pretrained(name, output_hidden_states=True,
                                                    cache_dir="cache/models/transformers")
    tok   = AutoTokenizer.from_pretrained(name, cache_dir="cache/models/transformers")
    M = embed_all(model, tok, male); F = embed_all(model, tok, female)
    S = embed_all(model, tok, stem); H = embed_all(model, tok, shape)
    sm, sf = pok_cosmetric(M, F, S)      # STEM
    hm, hf = pok_cosmetric(M, F, H)      # SHAPE
    p = PUB[name]
    rows.append({"model":name.split("/")[-1],
                 "STEM_m":round(sm,4),"pub":p[0],"Δ":round(sm-p[0],4),
                 "STEM_f":round(sf,4),"pub_f":p[1],"Δf":round(sf-p[1],4),
                 "SHAPE_m":round(hm,4),"pub_sm":round(p[2],4),"Δsm":round(hm-p[2],4),
                 "SHAPE_f":round(hf,4),"pub_sf":round(p[3],4),"Δsf":round(hf-p[3],4)})
    print(rows[-1])
    del model, tok; gc.collect(); torch.cuda.empty_cache()

cosdf = pd.DataFrame(rows); cosdf.to_csv("cos_replication.csv", index=False)
print("\n", cosdf.to_string(index=False))

male    lines= 57  unique= 55  dup=  2
female  lines= 56  unique= 55  dup=  1
stem    lines= 48  unique= 48  dup=  0
shape   lines= 77  unique= 74  dup=  3


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{'model': 'banglabert', 'STEM_m': 0.8586, 'pub': 0.8586, 'Δ': -0.0, 'STEM_f': 0.8607, 'pub_f': 0.8607, 'Δf': 0.0, 'SHAPE_m': 0.8884, 'pub_sm': 0.8884, 'Δsm': -0.0, 'SHAPE_f': 0.8895, 'pub_sf': 0.8895, 'Δsf': 0.0}


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{'model': 'banglishbert', 'STEM_m': 0.9161, 'pub': 0.9161, 'Δ': -0.0, 'STEM_f': 0.9189, 'pub_f': 0.9189, 'Δf': 0.0, 'SHAPE_m': 0.9362, 'pub_sm': 0.9362, 'Δsm': -0.0, 'SHAPE_f': 0.9387, 'pub_sf': 0.9387, 'Δsf': -0.0}


...

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

[transformers] BertForPreTraining LOAD REPORT from: Kowsher/bangla-bert
Key                         | Status  | 
----------------------------+---------+-
cls.seq_relationship.weight | MISSING | 
cls.seq_relationship.bias   | MISSING | 
bert.pooler.dense.weight    | MISSING | 
bert.pooler.dense.bias      | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'model': 'bangla-bert', 'STEM_m': 0.5034, 'pub': 0.5034, 'Δ': -0.0, 'STEM_f': 0.4962, 'pub_f': 0.4962, 'Δf': -0.0, 'SHAPE_m': 0.5225, 'pub_sm': 0.5225, 'Δsm': 0.0, 'SHAPE_f': 0.5124, 'pub_sf': 0.5124, 'Δsf': 0.0}


Loading weights:   0%|          | 0/34 [00:00<?, ?it/s]

{'model': 'sahajBERT', 'STEM_m': 0.7008, 'pub': 0.7008, 'Δ': 0.0, 'STEM_f': 0.6941, 'pub_f': 0.6941, 'Δf': 0.0, 'SHAPE_m': 0.729, 'pub_sm': 0.729, 'Δsm': 0.0, 'SHAPE_f': 0.7241, 'pub_sf': 0.7241, 'Δsf': -0.0}

            model  STEM_m    pub    Δ  STEM_f  pub_f   Δf  SHAPE_m  pub_sm  Δsm  SHAPE_f  pub_sf  Δsf
      banglabert  0.8586 0.8586 -0.0  0.8607 0.8607  0.0   0.8884  0.8884 -0.0   0.8895  0.8895  0.0
    banglishbert  0.9161 0.9161 -0.0  0.9189 0.9189  0.0   0.9362  0.9362 -0.0   0.9387  0.9387 -0.0
bangla-bert-base  0.5416 0.5416  0.0  0.5896 0.5896 -0.0   0.5591  0.5591 -0.0   0.5863  0.5863 -0.0
     bangla-bert  0.5034 0.5034 -0.0  0.4962 0.4962 -0.0   0.5225  0.5225  0.0   0.5124  0.5124  0.0
       sahajBERT  0.7008 0.7008  0.0  0.6941 0.6941  0.0   0.7290  0.7290  0.0   0.7241  0.7241 -0.0
